In [ ]:
!nvidia-smi

Sun May  3 09:59:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Розпаковую дані в папку custom_data

In [ ]:
!unzip -q /content/data.zip -d /content/custom_data

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/SpaceNamee/Computer-Vision-1/master/train_val_split.py

!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

--2026-05-03 10:19:12--  https://raw.githubusercontent.com/SpaceNamee/Computer-Vision-1/master/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-05-03 10:19:12 (53.7 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Created folder at /content/data/train/images
Created folder at /content/data/train/labels
Created folder at /content/data/validation/images
Created folder at /content/data/validation/labels
Number of image files: 162
Number of annotation files: 162
Images moving to train: 145
Images moving to validation: 17


Next, we'll install the Ultralytics library in this Google Colab instance. This Python library will be used to train the YOLO model.


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.8 MB/s eta 0:00:00


There's one last step before we can run training: we need to create the Ultralytics training configuration YAML file. This file specifies the location of your train and validation data, and it also defines the model's classes.

Run the code block below to automatically generate a `data.yaml` configuration file.

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
  # Read class.txt to get class name
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return

  with open(path_to_classes_txt, "r") as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())

  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      "path": '/content/data',
      "train": "train/images",
      "val": "validation/images",
      "nc": number_of_classes,
      "names": classes
  }

  # Write data to Yaml file
  with open(path_to_data_yaml, "w") as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml



Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 11
names:
- MMs_peanut
- MMs_regular
- airheads
- gummy_worms
- milky_way
- nerds
- skittles
- snickers
- starbust
- three_musketeers
- twizzlers


In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, i

The model has been trained; now it's time to test it! The commands below run the model on the images in the validation folder and then display the results for the first 10 images

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,417,057 parameters, 0 gradients, 21.3 GFLOPs

image 1/17 /content/data/validation/images/1a0a0a87-candy_138.jpg: 384x640 1 MMs_peanut, 59.6ms
image 2/17 /content/data/validation/images/40b644e3-candy_9.jpg: 384x640 1 airheads, 1 gummy_worms, 1 nerds, 1 twizzlers, 10.8ms
image 3/17 /content/data/validation/images/5d97f5cd-candy_70.jpg: 480x640 2 MMs_peanuts, 1 airheads, 1 milky_way, 1 skittles, 1 three_musketeers, 60.4ms
image 4/17 /content/data/validation/images/66628b3d-candy_146.jpg: 384x640 1 airheads, 11.4ms
image 5/17 /content/data/validation/images/79241d26-candy_10.jpg: 384x640 1 MMs_peanut, 1 airheads, 1 snickers, 1 starbust, 10.8ms
image 6/17 /content/data/validation/images/84f6fae9-candy_86.jpg: 480x640 1 MMs_peanut, 1 airheads, 1 skittles, 1 snickers, 1 twizzlers, 13.0ms
image 7/17 /content/data/validation/images/888bff06-candy_93.jpg: 480x640 2 airheadss

In [ ]:
!yolo detect val model=runs/detect/train/weights/best.pt data=data.yaml

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,417,057 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2777.4±542.3 MB/s, size: 280.1 KB)
val: Scanning /content/data/validation/labels.cache... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17 2.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s
                   all         17         59      0.958       0.97       0.98      0.891
            MMs_peanut          5          5      0.991          1      0.995        0.9
           MMs_regular          1          1      0.936          1      0.995      0.895
              airheads         13         17      0.997          1      0.995      0.908
           gummy_worms          3          3      0.733          1      0.995      0.929
             milky_way          3          4    

In [ ]:
!yolo detect val model=runs/detect/train/weights/best.pt data=data.yaml save_json=True plots=True

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,417,057 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3440.2±410.9 MB/s, size: 187.0 KB)
val: Scanning /content/data/validation/labels.cache... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17 3.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.4it/s 0.8s
                   all         17         59      0.958       0.97       0.98      0.891
            MMs_peanut          5          5      0.991          1      0.995        0.9
           MMs_regular          1          1      0.936          1      0.995      0.895
              airheads         13         17      0.997          1      0.995      0.908
           gummy_worms          3          3      0.733          1      0.995      0.929
             milky_way          3          4    

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

files = sorted(glob.glob('/content/runs/detect/predict/*.jpg'))[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, image_path in zip(axes.flatten(), files):
    img = mpimg.imread(image_path)
    ax.imshow(img)
    ax.set_title(image_path.split('/')[-1], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Create "my_model" folder to store model weights and train results
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train
%cd /content

/content/my_model
  adding: my_model.pt (deflated 8%)
  adding: train/ (stored 0%)
  adding: train/confusion_matrix_normalized.png (deflated 21%)
  adding: train/val_batch0_pred.jpg (deflated 8%)
  adding: train/train_batch2.jpg (deflated 5%)
  adding: train/train_batch500.jpg (deflated 8%)
  adding: train/BoxP_curve.png (deflated 10%)
  adding: train/train_batch1.jpg (deflated 7%)
  adding: train/weights/ (stored 0%)
  adding: train/weights/best.pt (deflated 8%)
  adding: train/weights/last.pt (deflated 8%)
  adding: train/train_batch0.jpg (deflated 7%)
  adding: train/results.png (deflated 7%)
  adding: train/train_batch501.jpg (deflated 14%)
  adding: train/BoxR_curve.png (deflated 11%)
  adding: train/labels.jpg (deflated 29%)
  adding: train/args.yaml (deflated 53%)
  adding: train/confusion_matrix.png (deflated 22%)
  adding: train/BoxF1_curve.png (deflated 7%)
  adding: train/train_batch502.jpg (deflated 13%)
  adding: train/results.csv (deflated 63%)
  adding: train/val_batch0_